# Automated Inventory Reconciliation Pipeline
## Kopikita Roastery — TechSprint 2026 | Data Automation

---

**Deskripsi:**  
Pipeline ini mengimplementasikan sistem rekonsiliasi inventaris otomatis untuk UMKM F&B Kopikita Roastery. Pipeline memproses data mentah dari tiga sumber (POS kasir, gudang, dan master katalog) menjadi laporan `Action_Report.csv` yang mengklasifikasikan status setiap bahan baku per hari sebagai **Safe**, **Restock**, atau **Anomaly**.

**Alur Eksekusi:**
1. Data Ingestion — unduh dan normalisasi data mentah
2. Data Cleansing — bersihkan noise dan perbaiki format
3. Calculation — BOM Calculation untuk konversi satuan POS → gudang
4. Anomaly Logic — deteksi shrinkage dengan hybrid MAD statistical method

**Strategi yang digunakan:** S02 (`neg_qty=quarantine`, `dup_nonid=drop`, `qty_suspicious=invalid`, `addinfo=ignore`)

## Setup & Dependencies

In [ ]:
import json
import re
import time
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
print('Libraries loaded successfully.')


---
## Section 1 — Data Ingestion

Tahap ini membaca dataset dari direktori lokal dan menormalisasi format file ke bentuk yang siap diproses.

**Konfigurasi path** didefinisikan di cell berikut — sesuaikan `DATA_DIR` dengan lokasi folder dataset Anda.

Sub-proses:
1. **Path Configuration** — mendefinisikan lokasi setiap file sumber
2. **JSON → CSV Conversion** — meratakan (flatten) struktur JSON nested menjadi tabel datar, sekaligus menangani:
   - **Schema drift** pada `warehouse_stock`: field `stock_remaining` berganti nama menjadi `sisa_stok_akhir` pada April 2025
   - **UoM mismatch**: beberapa baris menggunakan satuan supplier (`kilogram`, `liter`, `karton`) alih-alih satuan warehouse (`gram`, `ml`, `pcs`)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA INGESTION — Path Configuration
# Sesuaikan DATA_DIR dengan lokasi folder dataset Anda
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR = Path('data')

# Raw source files
SALES_PATH      = DATA_DIR / 'sales_history (Competitors).csv'
WAREHOUSE_PATH  = DATA_DIR / 'warehouse_stock (Competitors).json'
BOM_PATH        = DATA_DIR / 'Recipe_BOM (Competitors).json'
INVENTORY_PATH  = DATA_DIR / 'Master_Inventory (Competitors).csv'
EMPLOYEE_PATH   = DATA_DIR / 'Employee (Competitors).json'

# Processed output files
SALES_FIXED_PATH     = DATA_DIR / 'sales_history_fixed.csv'
WAREHOUSE_CSV_PATH   = DATA_DIR / 'warehouse_stock.csv'
BOM_CSV_PATH         = DATA_DIR / 'Recipe_BOM.csv'
EMPLOYEE_CSV_PATH    = DATA_DIR / 'Employee.csv'

# Final output
ACTION_REPORT_PATH = Path('Action_Report.csv')

# Verify semua file sumber tersedia
missing = [p for p in [SALES_PATH, WAREHOUSE_PATH, BOM_PATH, INVENTORY_PATH, EMPLOYEE_PATH] if not p.exists()]
if missing:
    print('FILE TIDAK DITEMUKAN:')
    for p in missing:
        print(f'  {p}')
else:
    print('Semua file sumber ditemukan:')
    for p in [SALES_PATH, WAREHOUSE_PATH, BOM_PATH, INVENTORY_PATH, EMPLOYEE_PATH]:
        print(f'  {p.name} ({p.stat().st_size / 1024:.1f} KB)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA INGESTION — Konversi JSON ke CSV dengan normalisasi
# ─────────────────────────────────────────────────────────────────────────────

# 1. Employee — struktur flat, langsung konversi
with open(EMPLOYEE_PATH) as f:
    emp = json.load(f)
pd.DataFrame(emp['employees']).to_csv(EMPLOYEE_CSV_PATH, index=False)

# 2. Recipe_BOM — flatten: 1 baris per ingredient per menu
with open(BOM_PATH) as f:
    bom_raw = json.load(f)
bom_rows = [
    {
        'Menu_ID': m['Menu_ID'], 'Menu_Name': m['Menu_Name'],
        'Selling_Price_IDR': m['Selling_Price_IDR'],
        'Item_ID': ing['Item_ID'], 'Item_Name': ing['Item_Name'],
        'qty_used': ing['qty_used'], 'UoM': ing['UoM'],
    }
    for m in bom_raw['menu_items'] for ing in m['ingredients']
]
pd.DataFrame(bom_rows).to_csv(BOM_CSV_PATH, index=False)

# 3. Warehouse stock — flatten + normalisasi
with open(WAREHOUSE_PATH) as f:
    wh_raw = json.load(f)

wh_rows = []
for rec in wh_raw['records']:
    for entry in rec['stock_entries']:
        # Schema drift: field berganti nama dari stock_remaining (Jan–Mar)
        # menjadi sisa_stok_akhir (Apr–Jun)
        stock = entry.get('stock_remaining', entry.get('sisa_stok_akhir'))
        wh_rows.append({
            'record_id': rec['record_id'], 'date': rec['date'],
            'recorded_by': rec['recorded_by'], 'Item_ID': entry['Item_ID'],
            'stock_remaining': stock, 'delivery_in': entry['delivery_in'],
            'UoM': entry['UoM'],
        })

wh_df = pd.DataFrame(wh_rows)

# UoM fix: normalisasi satuan supplier ke satuan warehouse
# kilogram → gram (×1000), liter → ml (×1000)
for uom_from, uom_to in [('kilogram', 'gram'), ('liter', 'ml')]:
    mask = wh_df['UoM'] == uom_from
    wh_df.loc[mask, ['stock_remaining', 'delivery_in']] *= 1000
    wh_df.loc[mask, 'UoM'] = uom_to

# Fix INV-0007 (Condensed Milk): 1 baris salah input karton (2025-01-17)
# Diverifikasi: hari sebelum dan sesudah bernilai 76800.0 ml, delivery_in=0
mask_karton = (wh_df['Item_ID'] == 'INV-0007') & (wh_df['UoM'] == 'karton')
wh_df.loc[mask_karton, 'stock_remaining'] = 76800.0
wh_df.loc[mask_karton, 'UoM'] = 'ml'

wh_df.to_csv(WAREHOUSE_CSV_PATH, index=False)

print('Konversi selesai.')
print(f'  Recipe_BOM.csv     : {len(bom_rows)} baris')
print(f'  warehouse_stock.csv: {len(wh_df)} baris')
print(f'  Employee.csv       : {len(emp["employees"])} baris')


---
## Section 2 — Data Cleansing

Data POS (`sales_history`) mengandung berbagai kelas kerusakan yang ditemukan saat eksplorasi:

| Masalah | Jumlah Baris | Penanganan |
|---|---|---|
| Transaction_ID malformed | 1,659 | Fix regex (strip whitespace, underscore→dash, remove trailing X) |
| DateTime format non-standar (6 format) | 3,030 | Multi-format parser |
| DateTime null (semua kolom juga null) | 1,008 | Drop |
| Quantity non-numerik (`two`, `dua`, `2,0`, `1 pcs`) | 2,035 | Parse word-to-number + strip suffix |
| Menu_ID null (fallback via Item_Name) | 1,287 | Mapping Item_Name → Menu_ID |
| Duplicate rows identik | 1,525 | Drop |

Output tahap ini: `sales_history_fixed.csv` — 168,080 baris, diurutkan ascending by DateTime.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CLEANSING — Fix Transaction_ID
# Tiga pola malformed: trailing whitespace, underscore+X, double dash
# ─────────────────────────────────────────────────────────────────────────────
sales = pd.read_csv(SALES_PATH)
print(f'Loaded {len(sales):,} baris')

def fix_trx_id(val):
    val = str(val).strip()
    val = re.sub(r'^TRX_', 'TRX-', val)   # TRX_YYYYMMDD-XXXXX → TRX-
    val = val.rstrip('X')                  # trailing X
    val = re.sub(r'^TRX--', 'TRX-', val)  # double dash
    return val

before = (~sales['Transaction_ID'].str.match(r'^TRX-\d{8}-\d{4}$', na=False)).sum()
sales['Transaction_ID'] = sales['Transaction_ID'].apply(fix_trx_id)
after  = (~sales['Transaction_ID'].str.match(r'^TRX-\d{8}-\d{4}$', na=False)).sum()
print(f'Transaction_ID: {before} malformed → {after} tersisa')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDA — Gambaran umum missing values sebelum cleaning
# ─────────────────────────────────────────────────────────────────────────────
sales_raw = pd.read_csv(SALES_PATH)

null_pct = sales_raw.isnull().mean() * 100
null_pct = null_pct.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.barh(null_pct.index, null_pct.values, color=PALETTE[0], edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('Missing Values (%)')
ax.set_title('Distribusi Missing Values per Kolom — sales_history (raw)', fontweight='bold')
ax.set_xlim(0, 110)
plt.tight_layout()
plt.show()
print(f'Total sel null: {sales_raw.isnull().sum().sum():,} dari {sales_raw.size:,} ({sales_raw.isnull().mean().mean()*100:.1f}%)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CLEANSING — Normalisasi DateTime (6 format berbeda)
# ─────────────────────────────────────────────────────────────────────────────
DATETIME_FORMATS = [
    '%Y-%m-%d %H:%M:%S',   # standard
    '%d/%m/%Y %H:%M',       # 14/01/2025 17:31
    '%b %d %Y %H:%M',       # Jun 13 2025 09:32
    '%Y%m%d%H%M',           # 202503270913
    '%m-%d-%Y %I:%M %p',    # 01-14-2025 02:26 PM
    '%Y.%m.%d %H:%M',       # 2025.03.05 08:47
]

def parse_datetime(val):
    if pd.isna(val):
        return pd.NaT
    for fmt in DATETIME_FORMATS:
        try:
            return pd.to_datetime(str(val).strip(), format=fmt)
        except ValueError:
            continue
    return pd.NaT

before_null = sales['DateTime'].isna().sum()
before_bad  = (pd.to_datetime(sales['DateTime'], errors='coerce').isna() & sales['DateTime'].notna()).sum()
sales['DateTime'] = sales['DateTime'].apply(parse_datetime)
after_null = sales['DateTime'].isna().sum()

# Drop baris null DateTime: diverifikasi semua kolom lain juga null
before_drop = len(sales)
sales = sales[sales['DateTime'].notna()].reset_index(drop=True)
print(f'DateTime: {before_bad} non-standar berhasil diparsing')
print(f'Dropped {before_drop - len(sales)} baris null DateTime ({len(sales):,} tersisa)')
sales['DateTime'] = sales['DateTime'].dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDA — Transaction_ID: distribusi pola malformed
# ─────────────────────────────────────────────────────────────────────────────
import re as _re

def classify_trx(val):
    val = str(val).strip()
    if _re.match(r'^TRX_\d{8}-\d{4}X$', val): return 'Underscore + X'
    if _re.match(r'^TRX--\d{8}-\d{4}$', val): return 'Double Dash'
    if not _re.match(r'^TRX-\d{8}-\d{4}$', val): return 'Trailing Whitespace'
    return 'Valid'

tmp = sales_raw.copy()
tmp['trx_class'] = tmp['Transaction_ID'].apply(classify_trx)
counts = tmp['trx_class'].value_counts()
malformed = counts[counts.index != 'Valid']

fig, ax = plt.subplots(figsize=(7, 2.8))
colors = [PALETTE[2], PALETTE[3], PALETTE[1]]
bars = ax.barh(malformed.index, malformed.values, color=colors, edgecolor='white')
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel('Jumlah Baris')
ax.set_title('Pola Transaction_ID Malformed (total 1.659 baris)', fontweight='bold')
ax.set_xlim(0, max(malformed.values) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CLEANSING — Normalisasi Quantity non-numerik
# Pemetaan: word-to-number (EN/ID), comma-decimal, strip unit suffix
# ─────────────────────────────────────────────────────────────────────────────
WORD_TO_NUM = {'two': 2, 'dua': 2, 'one': 1, 'satu': 1, 'three': 3, 'tiga': 3}

def fix_quantity(val):
    if pd.isna(val):
        return val
    s = str(val).strip().lower()
    if s in WORD_TO_NUM:
        return float(WORD_TO_NUM[s])
    s = re.sub(r'\s*(cups?|pcs?|porsi|unit)$', '', s).strip()
    s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return val

before = pd.to_numeric(sales['Quantity'], errors='coerce').isna().sum()
sales['Quantity'] = sales['Quantity'].apply(fix_quantity)
sales['Quantity'] = pd.to_numeric(sales['Quantity'], errors='coerce')
after = sales['Quantity'].isna().sum()
print(f'Quantity: {before - after} non-numerik berhasil diparsing, {after} null tersisa')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDA — DateTime: distribusi format & volume transaksi per bulan
# ─────────────────────────────────────────────────────────────────────────────
_FORMATS = {
    '%Y-%m-%d %H:%M:%S': 'Standar (YYYY-MM-DD)',
    '%d/%m/%Y %H:%M':    'DD/MM/YYYY HH:MM',
    '%b %d %Y %H:%M':    'Mon DD YYYY HH:MM',
    '%Y%m%d%H%M':        'YYYYMMDDHHmm',
    '%m-%d-%Y %I:%M %p': 'MM-DD-YYYY HH:MM AM/PM',
    '%Y.%m.%d %H:%M':    'YYYY.MM.DD HH:MM',
}
def detect_fmt(val):
    if pd.isna(val): return 'Null'
    for fmt, label in _FORMATS.items():
        try:
            pd.to_datetime(str(val).strip(), format=fmt)
            return label
        except: pass
    return 'Unknown'

sample = sales_raw['DateTime'].sample(3000, random_state=42)
fmt_counts = sample.apply(detect_fmt).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart format distribution
wedges, texts, autotexts = axes[0].pie(
    fmt_counts.values, labels=fmt_counts.index,
    autopct=lambda p: f'{p:.1f}%' if p > 0.5 else '',
    colors=PALETTE + ['#6B9E78', '#E8A838'],
    startangle=140, pctdistance=0.75
)
axes[0].set_title('Distribusi Format DateTime\n(sampel 3.000 baris)', fontweight='bold')

# Monthly volume after fix
sales_dt = pd.read_csv(SALES_FIXED_PATH, parse_dates=['DateTime'])
monthly = sales_dt.groupby(sales_dt['DateTime'].dt.to_period('M')).size()
monthly.index = monthly.index.astype(str)
axes[1].bar(monthly.index, monthly.values, color=PALETTE[0], edgecolor='white')
axes[1].set_title('Volume Transaksi per Bulan\n(setelah normalisasi)', fontweight='bold')
axes[1].set_xlabel('Bulan')
axes[1].set_ylabel('Jumlah Transaksi')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CLEANSING — Fallback Menu_ID null via Item_Name
# Diverifikasi: 1,287 baris null Menu_ID punya Item_Name valid,
# dan 100% berhasil di-mapping ke Menu_ID yang ada di BOM
# ─────────────────────────────────────────────────────────────────────────────
bom_df = pd.read_csv(BOM_CSV_PATH)
name_to_menu = dict(zip(
    bom_df['Menu_Name'].str.lower().str.strip(),
    bom_df['Menu_ID']
))

before = sales['Menu_ID'].isna().sum()
mask   = sales['Menu_ID'].isna() & sales['Item_Name'].notna()
sales.loc[mask, 'Menu_ID'] = (
    sales.loc[mask, 'Item_Name'].str.lower().str.strip().map(name_to_menu)
)
after = sales['Menu_ID'].isna().sum()
print(f'Menu_ID: {before - after} null di-recover via Item_Name, {after} null tersisa')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDA — Quantity: non-numerik dan suspicious values (99 & 150)
# ─────────────────────────────────────────────────────────────────────────────
qty_raw = pd.to_numeric(sales_raw['Quantity'], errors='coerce')
non_num = sales_raw[qty_raw.isna() & sales_raw['Quantity'].notna()]['Quantity'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Non-numerik frequency
axes[0].barh(non_num.index, non_num.values, color=PALETTE[1], edgecolor='white')
axes[0].bar_label(axes[0].containers[0], padding=3, fontsize=9)
axes[0].set_title('Varian Quantity Non-Numerik', fontweight='bold')
axes[0].set_xlabel('Frekuensi')

# 2. Quantity distribution (valid range 1-10) — histogram
qty_valid = qty_raw.dropna()
qty_valid = qty_valid[(qty_valid > 0) & (qty_valid <= 200)]
axes[1].hist(qty_valid, bins=50, color=PALETTE[0], edgecolor='white', log=True)
axes[1].axvline(99,  color=PALETTE[3], linewidth=2, linestyle='--', label='Suspicious (99)')
axes[1].axvline(150, color=PALETTE[2], linewidth=2, linestyle='--', label='Suspicious (150)')
axes[1].set_title('Distribusi Quantity (log-scale)', fontweight='bold')
axes[1].set_xlabel('Quantity')
axes[1].set_ylabel('Frekuensi (log)')
axes[1].legend(fontsize=8)

# 3. Heatmap Qty suspicious × Menu_ID (sample)
susp = sales_raw[qty_raw.isin([99, 150])][['Menu_ID','Quantity']].dropna()
pivot = susp.groupby(['Menu_ID','Quantity']).size().unstack(fill_value=0)
if not pivot.empty:
    im = axes[2].imshow(pivot.T.values, aspect='auto', cmap='Oranges')
    axes[2].set_xticks(range(len(pivot.index)))
    axes[2].set_xticklabels(pivot.index, rotation=90, fontsize=6)
    axes[2].set_yticks(range(len(pivot.columns)))
    axes[2].set_yticklabels(pivot.columns.astype(int))
    axes[2].set_title('Frekuensi Qty 99/150 per Menu\n(merata = artifact POS)', fontweight='bold')
    plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CLEANSING — Drop duplicates identik & sort by DateTime
# ─────────────────────────────────────────────────────────────────────────────
before = len(sales)
sales  = sales.drop_duplicates().reset_index(drop=True)
print(f'Dropped {before - len(sales)} duplicate rows identik')

sales = sales.sort_values('DateTime', ascending=True, na_position='last').reset_index(drop=True)
sales.to_csv(SALES_FIXED_PATH, index=False)
print(f'Saved sales_history_fixed.csv — {len(sales):,} baris')
print(f'  Rentang: {sales["DateTime"].iloc[0]} → {sales["DateTime"].iloc[-1]}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDA — Menu_ID: null recovery funnel & ghost ID distribution
# ─────────────────────────────────────────────────────────────────────────────
sales_tmp = pd.read_csv(SALES_FIXED_PATH)
total_null_before = sales_raw['Menu_ID'].isna().sum()
with_item_name    = sales_raw[sales_raw['Menu_ID'].isna() & sales_raw['Item_Name'].notna()].shape[0]
VALID_MENUS_VIZ   = {f'MENU-{str(i).zfill(3)}' for i in range(1,26)}
ghost             = sales_tmp[sales_tmp['Menu_ID'].notna() & ~sales_tmp['Menu_ID'].isin(VALID_MENUS_VIZ)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Funnel
stages   = ['Menu_ID Null\n(total)', 'Punya Item_Name\nValid', 'Berhasil Di-mapping', 'Null Tersisa']
values   = [total_null_before, with_item_name, with_item_name, 0]
colors_f = [PALETTE[3], PALETTE[2], PALETTE[0], '#CCCCCC']
bars     = axes[0].barh(stages, values, color=colors_f, edgecolor='white')
axes[0].bar_label(bars, padding=4, fontsize=9)
axes[0].set_title('Funnel Recovery Menu_ID Null', fontweight='bold')
axes[0].set_xlabel('Jumlah Baris')
axes[0].invert_yaxis()

# Ghost ID distribution
ghost_counts = ghost['Menu_ID'].value_counts()
axes[1].bar(ghost_counts.index, ghost_counts.values, color=PALETTE[3], edgecolor='white')
axes[1].set_title('Distribusi Ghost Menu_ID\n(tidak dapat di-recover)', fontweight='bold')
axes[1].set_xlabel('Menu_ID')
axes[1].set_ylabel('Frekuensi')
for bar, val in zip(axes[1].patches, ghost_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontsize=9)

plt.tight_layout()
plt.show()


---
## Section 3 — Calculation: BOM Calculation

Tahap ini mengkonversi data penjualan POS (dalam satuan "cup/porsi per menu") menjadi konsumsi bahan baku (dalam satuan gram/ml/pcs) menggunakan tabel resep `Recipe_BOM`.

**Strategi S02** yang diterapkan pada tahap cleansing:
- Negatif Quantity → **quarantine** (tidak diasumsikan sebagai retur)
- Quantity 99 atau 150 → **quarantine** (terbukti artifact sistem POS — menyebabkan 625 false positive Type 1 jika dibiarkan)
- Ghost Menu_ID → **quarantine**
- Duplicate non-identik → **ambil pertama** (pilihan paling netral)
- Additional_Info error string → **ignore** (terbukti menghilangkan shrinkage nyata jika difilter)

**Formula BOM Calculation:**
$$\text{consumed}_{(\text{date}, \text{item})} = \sum_{\text{transaksi valid}} \text{Quantity} \times \text{qty\_used}_{(\text{menu, item})}$$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CALCULATION — Load data bersih & preprocessing warehouse
# ─────────────────────────────────────────────────────────────────────────────
sales_clean = pd.read_csv(SALES_FIXED_PATH)
sales_clean['DateTime'] = pd.to_datetime(sales_clean['DateTime'], errors='coerce')
sales_clean['Date']     = sales_clean['DateTime'].dt.date
sales_clean['Quantity'] = pd.to_numeric(sales_clean['Quantity'], errors='coerce')

bom_df = pd.read_csv(BOM_CSV_PATH)[['Menu_ID', 'Item_ID', 'qty_used']]
inv_df = pd.read_csv(INVENTORY_PATH)

# Preprocessing warehouse: pre-compute expected consumption
wh = pd.read_csv(WAREHOUSE_CSV_PATH)
wh['date'] = pd.to_datetime(wh['date']).dt.date
wh = wh.sort_values(['Item_ID', 'date']).reset_index(drop=True)

wh['prev_date']  = wh.groupby('Item_ID')['date'].shift(1)
wh['prev_stock'] = wh.groupby('Item_ID')['stock_remaining'].shift(1)

# Hanya hitung expected_consumption untuk hari berurutan
# Hari setelah gap tanggal (6 hari bolong) di-skip agar tidak terjadi
# false Anomaly akibat dua hari konsumsi terakumulasi dalam satu selisih
wh['day_delta'] = (
    pd.to_datetime(wh['date'].astype(str)) -
    pd.to_datetime(wh['prev_date'].astype(str))
).dt.days
wh['expected_consumption'] = np.where(
    wh['day_delta'] == 1,
    wh['prev_stock'] + wh['delivery_in'] - wh['stock_remaining'],
    np.nan
)

print('Warehouse preprocessing selesai.')
print(f'  Total rows: {len(wh):,}')
print(f'  Rows dengan expected_consumption valid: {wh["expected_consumption"].notna().sum():,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CALCULATION — Hitung Restock threshold per item
# Non-Karton: Min_Stock_Threshold × konversi satuan
# Karton: avg(stock_before_delivery) pada hari ada delivery
#         → threshold empiris dari perilaku restock historis
# ─────────────────────────────────────────────────────────────────────────────
UOM_MULTIPLIER = {'Kilogram': 1000, 'Liter': 1000, 'Pcs': 1, 'Karton': 1}
inv_df['restock_threshold'] = inv_df.apply(
    lambda r: float(r['Min_Stock_Threshold'] * UOM_MULTIPLIER[r['Supplier_UoM']]),
    axis=1
)

# Derivasi empiris threshold Karton dari event delivery aktual
wh['stock_before_delivery'] = wh['stock_remaining'] - wh['delivery_in']
karton_ids  = set(inv_df[inv_df['Supplier_UoM'] == 'Karton']['Item_ID'])
deliveries  = wh[wh['Item_ID'].isin(karton_ids) & (wh['delivery_in'] > 0)]
avg_before  = deliveries.groupby('Item_ID')['stock_before_delivery'].mean()
for item_id, thr in avg_before.items():
    inv_df.loc[inv_df['Item_ID'] == item_id, 'restock_threshold'] = round(thr, 1)

threshold_map = inv_df.set_index('Item_ID')['restock_threshold'].to_dict()
print('Restock threshold per item (sample):')
print(inv_df[['Item_ID','Item_Name','Supplier_UoM','Min_Stock_Threshold','restock_threshold']].head(10).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CALCULATION — Terapkan strategi S02 & BOM Calculation
# Quarantine: negatif, null/zero, ghost Menu_ID, qty suspicious (99/150)
# ─────────────────────────────────────────────────────────────────────────────
VALID_MENUS = {f'MENU-{str(i).zfill(3)}' for i in range(1, 26)}

df = sales_clean.copy()
df['_invalid'] = False

# Quarantine negatif
df.loc[df['Quantity'] < 0, '_invalid'] = True
# Quarantine null / zero
df.loc[df['Quantity'].isna() | (df['Quantity'] == 0), '_invalid'] = True
# Quarantine ghost Menu_ID
df.loc[~df['Menu_ID'].isin(VALID_MENUS), '_invalid'] = True
# Quarantine quantity suspicious (99 dan 150)
# Terbukti menyebabkan 625 false positive Type 1 (POS > Warehouse)
# yang merupakan kebalikan dari shrinkage nyata
df.loc[df['Quantity'].isin([99, 150]), '_invalid'] = True

invalid_count = int(df['_invalid'].sum())
valid = df[~df['_invalid']].copy()

# Resolve duplicate Transaction_ID non-identik: ambil kemunculan pertama
valid = valid.drop_duplicates(subset='Transaction_ID', keep='first')

# BOM Calculation: setiap transaksi valid × ingredient dari resep
exploded = valid.merge(bom_df, on='Menu_ID', how='inner')
exploded['consumed'] = exploded['qty_used'] * exploded['Quantity']

# Agregasi: total konsumsi POS per (tanggal, item)
daily_consumption = (
    exploded.groupby(['Date', 'Item_ID'])['consumed']
    .sum()
    .reset_index()
    .rename(columns={'consumed': 'pos_consumed'})
)

print(f'Transaksi dikarantina : {invalid_count:,}')
print(f'Transaksi valid       : {len(valid):,}')
print(f'Daily consumption rows: {len(daily_consumption):,} (date × item combinations)')

---
## Section 4 — Anomaly Logic: Stock Reconciliation

Rekonsiliasi membandingkan konsumsi POS (`pos_consumed`) dengan konsumsi fisik yang tercatat di gudang (`expected_consumption`).

**Formula:**
$$\text{expected\_consumption}_t = \text{prev\_stock}_{t-1} + \text{delivery\_in}_t - \text{stock\_remaining}_t$$

$$\text{gap}_t = |\text{pos\_consumed}_t - \text{expected\_consumption}_t|$$

**Klasifikasi (priority order):**

$$\text{Anomaly} \succ \text{Restock} \succ \text{Safe}$$

**Restock** menggunakan `stock_before_delivery` bukan `stock_remaining`, karena pada hari delivery masuk, stok akhir sudah mencakup kiriman — yang relevan adalah kondisi *sebelum* kiriman tiba:

$$\text{stock\_before\_delivery} = \text{stock\_remaining} - \text{delivery\_in} = \text{prev\_stock} - \text{consumed}$$

**Anomaly Detection — Hybrid MAD (Modified Z-score):**

Dua lapis yang harus terpenuhi bersama:
1. **Hard floor:** $\text{gap} > 1.000$ (sesuai spec kompetisi)
2. **Statistical envelope per-item:** $\text{gap} > \text{median}(\text{gap}) + \frac{3.5 \times \text{MAD}}{0.6745}$

MAD dipilih atas IQR dan mean±σ karena *breakdown point* 50% — statistik tidak terpengaruh meskipun hingga 50% data terkontaminasi anomali.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ANOMALY LOGIC — Stock Reconciliation & Klasifikasi
# ─────────────────────────────────────────────────────────────────────────────
ANOMALY_THRESHOLD = 1_000   # hard floor sesuai spec kompetisi

# Merge konsumsi POS dengan data warehouse
recon = wh.rename(columns={'date': 'Date'}).merge(
    daily_consumption, on=['Date', 'Item_ID'], how='left'
)
recon['pos_consumed'] = recon['pos_consumed'].fillna(0)
recon['gap']          = (recon['pos_consumed'] - recon['expected_consumption']).abs()

# Restock menggunakan stok sebelum delivery masuk
recon['stock_before_delivery'] = recon['stock_remaining'] - recon['delivery_in']
recon['restock_threshold']     = recon['Item_ID'].map(threshold_map)

# ── Hybrid MAD Statistical Anomaly Detection ─────────────────────────────────
# Layer 1: gap > ANOMALY_THRESHOLD (hard floor)
# Layer 2: Modified Z-score > 3.5 per item
#          upper_fence = median + 3.5 × MAD / 0.6745
computable = recon[recon['gap'].notna()]

item_stats = computable.groupby('Item_ID')['gap'].agg(
    _median='median',
    _mad=lambda x: (x - x.median()).abs().median(),
).reset_index()
item_stats['mad_upper'] = np.where(
    item_stats['_mad'] > 0,
    item_stats['_median'] + 3.5 * item_stats['_mad'] / 0.6745,
    np.inf   # MAD=0: semua gap identik, cukup hard floor
)

recon = recon.merge(item_stats[['Item_ID', 'mad_upper']], on='Item_ID', how='left')

hard_floor   = recon['gap'].notna() & (recon['gap'] > ANOMALY_THRESHOLD)
stat_outlier = recon['gap'] > recon['mad_upper']
anomaly_cond = hard_floor & stat_outlier

# Klasifikasi: Anomaly > Restock > Safe
conditions = [
    anomaly_cond,
    recon['stock_before_delivery'] < recon['restock_threshold'],
]
recon['Action_Status'] = np.select(conditions, ['Anomaly', 'Restock'], default='Safe')

# Ringkasan hasil klasifikasi
counts = recon['Action_Status'].value_counts()
total  = len(recon)
print('=== Hasil Klasifikasi (7.350 baris = 175 hari × 42 item) ===')
for status, count in counts.items():
    print(f'  {status:<15}: {count:>5} ({count/total*100:.1f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDA — Warehouse: schema drift & restock threshold validation
# ─────────────────────────────────────────────────────────────────────────────
wh_viz = pd.read_csv(WAREHOUSE_CSV_PATH, parse_dates=['date'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Schema drift — INV-0001 time series dengan annotation
item_ts = wh_viz[wh_viz['Item_ID'] == 'INV-0001'].sort_values('date')
axes[0].plot(item_ts['date'], item_ts['stock_remaining'], color=PALETTE[0], linewidth=1.5)
drift_date = pd.Timestamp('2025-04-01')
axes[0].axvline(drift_date, color=PALETTE[3], linewidth=2, linestyle='--')
axes[0].text(drift_date, item_ts['stock_remaining'].max() * 0.95,
             ' Schema Drift\n 1 Apr 2025', color=PALETTE[3], fontsize=8)
axes[0].set_title('INV-0001 Espresso Bean — Stok Harian\n(schema drift ditangani, data kontinu)', fontweight='bold')
axes[0].set_xlabel('Tanggal')
axes[0].set_ylabel('Stock Remaining (gram)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b'))

# Restock threshold validation: scatter stock_before_delivery vs threshold
wh_viz['stock_before_delivery'] = wh_viz['stock_remaining'] - wh_viz['delivery_in']
inv_viz = pd.read_csv(INVENTORY_PATH)
UOM_M = {'Kilogram': 1000, 'Liter': 1000, 'Pcs': 1, 'Karton': 1}
inv_viz['thr'] = inv_viz.apply(lambda r: float(r['Min_Stock_Threshold'] * UOM_M[r['Supplier_UoM']]), axis=1)
karton_ids_v = set(inv_viz[inv_viz['Supplier_UoM']=='Karton']['Item_ID'])
dels_v = wh_viz[wh_viz['Item_ID'].isin(karton_ids_v) & (wh_viz['delivery_in'] > 0)]
for iid, thr in dels_v.groupby('Item_ID')['stock_before_delivery'].mean().items():
    inv_viz.loc[inv_viz['Item_ID']==iid, 'thr'] = round(thr, 1)
thr_map = inv_viz.set_index('Item_ID')['thr'].to_dict()
wh_viz['thr'] = wh_viz['Item_ID'].map(thr_map)
wh_del = wh_viz[wh_viz['delivery_in'] > 0].copy()
wh_nodel = wh_viz[wh_viz['delivery_in'] == 0].copy()
axes[1].scatter(wh_nodel['thr'], wh_nodel['stock_before_delivery'],
                alpha=0.2, s=8, color='#AAAAAA', label='Tidak ada delivery')
axes[1].scatter(wh_del['thr'], wh_del['stock_before_delivery'],
                alpha=0.5, s=12, color=PALETTE[0], label='Ada delivery')
lim = max(wh_viz['thr'].max(), wh_viz['stock_before_delivery'].clip(upper=50000).max())
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=1, label='y = x (threshold)')
axes[1].set_title('Validasi Threshold Restock\n(titik biru di bawah garis = kritis → delivery)', fontweight='bold')
axes[1].set_xlabel('Restock Threshold')
axes[1].set_ylabel('Stock Before Delivery')
axes[1].legend(fontsize=8)
axes[1].set_xlim(0)
axes[1].set_ylim(-5000)

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDA — Anomaly: distribusi gap, tipe anomali, dan kalender heatmap
# ─────────────────────────────────────────────────────────────────────────────
action = pd.read_csv(ACTION_REPORT_PATH, parse_dates=['Date'])
recon_viz = recon.copy()
recon_viz['direction'] = np.sign(recon_viz['pos_consumed'] - recon_viz['expected_consumption'])

fig = plt.figure(figsize=(14, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.4)

# 1. Box plot gap per item (top 10 by median gap)
ax1 = fig.add_subplot(gs[0, :2])
top_items = recon_viz[recon_viz['gap'].notna()].groupby('Item_ID')['gap'].median().nlargest(10).index
gap_data  = [recon_viz[recon_viz['Item_ID']==i]['gap'].dropna().values for i in top_items]
bp = ax1.boxplot(gap_data, patch_artist=True, medianprops={'color':'red','linewidth':2})
for patch in bp['boxes']:
    patch.set_facecolor('#2E86AB33')
ax1.set_xticks(range(1, len(top_items)+1))
ax1.set_xticklabels(top_items, rotation=45, ha='right', fontsize=8)
ax1.set_yscale('log')
ax1.set_ylabel('Gap (log scale)')
ax1.set_title('Distribusi Gap per Item (Top 10 by Median)\nJustifikasi: threshold per-item diperlukan', fontweight='bold')
# Mark MAD fences
for j, item in enumerate(top_items):
    g = recon_viz[recon_viz['Item_ID']==item]['gap'].dropna()
    mad = (g - g.median()).abs().median()
    fence = g.median() + 3.5*mad/0.6745 if mad > 0 else float('inf')
    if fence < 1e7:
        ax1.plot(j+1, fence, 'r^', markersize=6)

# 2. Scatter: pos_consumed vs expected (Anomaly direction)
ax2 = fig.add_subplot(gs[0, 2])
anom_rows = recon_viz[recon_viz['Action_Status'] == 'Anomaly'].copy()
t1 = anom_rows[anom_rows['direction'] > 0]
t2 = anom_rows[anom_rows['direction'] < 0]
lim_val = max(anom_rows[['pos_consumed','expected_consumption']].max().max(), 1)
ax2.scatter(t1['expected_consumption'], t1['pos_consumed'], color=PALETTE[2],
            s=40, label=f'Type 1 (n={len(t1)})', zorder=3)
ax2.scatter(t2['expected_consumption'], t2['pos_consumed'], color=PALETTE[3],
            s=40, label=f'Type 2 shrinkage (n={len(t2)})', zorder=3)
ax2.plot([0, lim_val], [0, lim_val], 'k--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Expected Consumption\n(Warehouse)')
ax2.set_ylabel('POS Consumed')
ax2.set_title('Arah Gap — Anomaly (S02)\nType 2 = Shrinkage Nyata', fontweight='bold')
ax2.legend(fontsize=8)

# 3. Calendar heatmap: Anomaly per (Date, Item)
ax3 = fig.add_subplot(gs[1, :])
anom_cal = anom_rows[['Date','Item_ID']].copy()
anom_cal['Date'] = pd.to_datetime(anom_cal['Date'])
anom_cal['Week'] = anom_cal['Date'].dt.isocalendar().week.astype(int)
anom_cal['DOW']  = anom_cal['Date'].dt.dayofweek
pivot_cal = anom_cal.pivot_table(index='Item_ID', columns='Week', aggfunc='size', fill_value=0)
if not pivot_cal.empty:
    im3 = ax3.imshow(pivot_cal.values, aspect='auto', cmap='YlOrRd',
                     interpolation='nearest')
    ax3.set_yticks(range(len(pivot_cal.index)))
    ax3.set_yticklabels(pivot_cal.index, fontsize=7)
    ax3.set_xlabel('Minggu ke-')
    ax3.set_title(f'Heatmap Kalender Anomaly (S02) — {len(anom_rows)} total | Item × Minggu', fontweight='bold')
    plt.colorbar(im3, ax=ax3, shrink=0.6, label='Jumlah Anomaly')

plt.suptitle('Analisis Anomaly Detection — Strategi S02', fontsize=12, fontweight='bold', y=1.01)
plt.show()


---
## Section 5 — Output: Action_Report.csv

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT — Generate Action_Report.csv
# 3 kolom wajib: Date, Item_ID, Action_Status
# ─────────────────────────────────────────────────────────────────────────────
action_report = (
    recon[['Date', 'Item_ID', 'Action_Status']]
    .assign(Date=lambda d: pd.to_datetime(d['Date']))
    .sort_values(['Date', 'Item_ID'])
    .reset_index(drop=True)
)

action_report.to_csv(ACTION_REPORT_PATH, index=False)
print('Action_Report.csv disimpan.')
print(f'Total baris : {len(action_report):,}')
print(f'Rentang Date: {action_report["Date"].min().date()} → {action_report["Date"].max().date()}')
print()
print('Preview:')
print(action_report.head(10).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT — Summary statistik & validasi
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 55)
print('RINGKASAN PIPELINE — Kopikita Roastery')
print('=' * 55)
print()
print('[ DATA INGESTION ]')
print(f'  Raw sales transactions : {len(sales_clean):>8,}')
print(f'  Transactions quarantined: {invalid_count:>7,}')
print(f'  Valid transactions     : {len(valid):>8,}')
print()
print('[ DATA CLEANSING ]')
print(f'  Transaction_ID fixed   : 1,659')
print(f'  DateTime normalized    : 3,030')
print(f'  Quantity parsed        : 2,035')
print(f'  Menu_ID recovered      : 1,287')
print(f'  Duplicates dropped     : 1,525')
print()
print('[ CALCULATION ]')
print(f'  BOM Calculation rows     : {len(daily_consumption):>8,}')
print(f'  Warehouse rows         : {len(wh):>8,}')
print(f'  Reconciliation rows    : {len(recon):>8,}')
print()
print('[ ANOMALY LOGIC ]')
counts = recon['Action_Status'].value_counts()
for s in ['Safe', 'Restock', 'Anomaly']:
    c = counts.get(s, 0)
    print(f'  {s:<20}: {c:>5} ({c/len(recon)*100:.2f}%)')
print()
print(f'[ OUTPUT ] Action_Report.csv — {len(action_report):,} baris')
print('=' * 55)

---
## Section 6 — Business Pipeline: Forecast, Inventory, MRP, Profitability, KPI, Insights

Mengeksekusi  untuk menghasilkan semua output hasil analitik bisnis dari data nyata.
Pipeline menghasilkan 6 file CSV/JSON yang digunakan oleh FastAPI backend.

In [ ]:
import sys, os, time

# Resolve dataset path relative to this notebook
nb_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
backend_dir = os.path.dirname(nb_dir)
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

from data_loader.generate_results import main as run_pipeline

t0 = time.time()
run_pipeline()
t1 = time.time()

import pandas as pd, json
dataset_dir = os.path.join(backend_dir, "dataset")

fc = pd.read_csv(os.path.join(dataset_dir, "forecast_result.csv"))
inv = pd.read_csv(os.path.join(dataset_dir, "inventory_result.csv"))
mrp = pd.read_csv(os.path.join(dataset_dir, "mrp_result.csv"))
pf = pd.read_csv(os.path.join(dataset_dir, "profitability_result.csv"))
kpi = pd.read_csv(os.path.join(dataset_dir, "kpi_result.csv"))
with open(os.path.join(dataset_dir, "insights.json")) as f:
    ins = json.load(f)

gross_margin = kpi.loc[kpi["kpi_name"]=="Gross Margin", "value"].values[0]
total_rev = kpi.loc[kpi["kpi_name"]=="Total Revenue", "value"].values[0]
total_profit = kpi.loc[kpi["kpi_name"]=="Total Profit", "value"].values[0]

print("=" * 55)
print("[ BUSINESS PIPELINE COMPLETE ]")
print("=" * 55)
print(f"  Runtime            : {t1-t0:.2f} seconds")
print()
print("[ OUTPUTS GENERATED ]")
print(f"  forecast_result.csv    : {len(fc):>3} menu items")
print(f"  inventory_result.csv   : {len(inv):>3} raw materials")
print(f"  mrp_result.csv         : {len(mrp):>3} BOM rows")
print(f"  profitability_result   : {len(pf):>3} products")
print(f"  kpi_result.csv         : {len(kpi):>3} KPI metrics")
print(f"  insights.json          : {len(ins):>3} business insights")
print()
print("[ KEY FINANCIALS (Base Scenario) ]")
print(f"  Total Revenue      : Rp {total_rev:>15,.0f}")
print(f"  Total Profit       : Rp {total_profit:>15,.0f}")
print(f"  Gross Margin       : {gross_margin:.1f}%")
print()
critical = inv[inv["status"]=="critical"]
print(f"  Critical Stock Items   : {len(critical)} items")
if len(critical) > 0:
    print(f"  Top Critical Item  : {critical.iloc[0]['product_name']}")
print("=" * 55)